# DIRE batch scoring (runs on Colab's free GPU)

Runs the real two-stage DIRE pipeline against a folder of images and produces a `dire_results.csv` you download and drop back into the local `ai-content-provenance-audit` project.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine).

**Stage 1** (`compute_dire.py`): reconstructs each image via a pretrained diffusion model, producing a DIRE (reconstruction-error) image.
**Stage 2** (equivalent to `demo.py`): classifies each DIRE image with a ResNet50 to get `Prob of being synthetic`.

Run the cells in order. Two manual steps are unavoidable: uploading your images, and uploading the classifier checkpoint (it's on a share-link drive, not a direct-download URL).

## 1. Install dependencies (MPI + Python packages)

In [ ]:
!apt-get -qq install -y libopenmpi-dev openmpi-bin > /dev/null
!pip -q install mpi4py opencv-python-headless blobfile einops tqdm

## 2. Clone DIRE (includes the `guided-diffusion` code, no submodule step needed)

In [ ]:
!git clone -q https://github.com/ZhendongWang6/DIRE.git
%cd DIRE/guided-diffusion
!pip -q install -e .

## 3. Download the diffusion checkpoint
Public OpenAI blob, no auth needed (~2GB, takes a few minutes).

In [ ]:
!mkdir -p models
!wget -q --show-progress -O models/256x256_diffusion_uncond.pt \
  https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt

## 4. Upload the DIRE classifier checkpoint (manual step)

This one isn't a direct-download URL — it's on the DIRE authors' share drive (these are the official links from their real GitHub README, not a random/spoofed source):
- BaiduDrive (password `dire`): https://pan.baidu.com/s/1Rdzc7l8P0RrJft0cW0a4Gg
- RecDrive (password `dire`): https://rec.ustc.edu.cn/share/ec980150-4615-11ee-be0a-eb822f25e070

**Security note:** download the file directly through your browser — don't install Baidu's desktop client if it's offered, you don't need it for a single-file download. Separately, `.pth` files are loaded via Python's `pickle` format, which can execute arbitrary code if a checkpoint is malicious — true for any PyTorch checkpoint from any source, not specific to this one. The load cell below uses `weights_only=True` to guard against that (it restricts loading to tensor data only). This whole notebook also runs in a disposable Colab VM, not your own machine, which limits the blast radius further.

Download the pretrained classifier `.pth` file from whichever loads for you, then run this cell and pick that file when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()
classifier_path = list(uploaded.keys())[0]
print(f"Using classifier checkpoint: {classifier_path}")

## 5. Upload the images to score

Zip up the images you want scored (e.g. the DIRE-flagged... well, at this stage, the ones you want DIRE to score in the first place) and upload the zip here.

In [ ]:
import zipfile, os
from google.colab import files

uploaded = files.upload()  # upload a .zip of images
zip_name = list(uploaded.keys())[0]

os.makedirs("input_images", exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("input_images")

# flatten in case the zip had a nested folder
image_files = []
for root, _, fnames in os.walk("input_images"):
    for fn in fnames:
        if fn.lower().endswith((".png", ".jpg", ".jpeg")):
            image_files.append(os.path.join(root, fn))
print(f"Found {len(image_files)} images")

## 6. Stage 1 — compute DIRE reconstruction images

Uses the same MODEL_FLAGS as the 256x256 unconditional model in `compute_dire.sh` / guided-diffusion's own README. `-n 1` since Colab gives you a single GPU (MPI is still required to satisfy the import, but runs as one process).

In [ ]:
MODEL_FLAGS = (
    "--attention_resolutions 32,16,8 --class_cond False --diffusion_steps 1000 "
    "--dropout 0.1 --image_size 256 --learn_sigma True --noise_schedule linear "
    "--num_channels 256 --num_head_channels 64 --num_res_blocks 2 "
    "--resblock_updown True --use_fp16 True --use_scale_shift_norm True"
)
SAMPLE_FLAGS = "--batch_size 4 --num_samples -1 --timestep_respacing ddim20 --use_ddim True"

!mpiexec -n 1 --allow-run-as-root python compute_dire.py \
  --model_path models/256x256_diffusion_uncond.pt \
  --images_dir input_images --recons_dir recons_out --dire_dir dire_out \
  --has_subfolder False {MODEL_FLAGS} {SAMPLE_FLAGS}

## 7. Stage 2 — classify the DIRE images, write dire_results.csv

Same model/preprocessing `demo.py` uses, but writing straight to CSV instead of parsing printed text.

In [ ]:
import csv, glob, sys
sys.path.insert(0, "..")  # DIRE/ root, for utils.utils and networks/
import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from utils.utils import get_network

model = get_network("resnet50")
# weights_only=True restricts unpickling to tensors/basic containers only,
# blocking arbitrary code execution if this checkpoint were ever tampered
# with — standard practice for loading any .pth from outside your own
# pipeline. If this errors on an unrecognized type, that's worth stopping
# and asking about rather than relaxing this flag.
state_dict = torch.load(classifier_path, map_location="cpu", weights_only=True)
if "model" in state_dict:
    state_dict = state_dict["model"]
model.load_state_dict(state_dict)
model.eval().cuda()

trans = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
])

dire_images = sorted(glob.glob("dire_out/**/*.png", recursive=True) + glob.glob("dire_out/**/*.jpg", recursive=True))
print(f"Scoring {len(dire_images)} DIRE images")

rows = []
with torch.no_grad():
    for path in dire_images:
        img = Image.open(path).convert("RGB")
        img = trans(img)
        img = TF.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        prob = model(img.unsqueeze(0).cuda()).sigmoid().item()
        filename = os.path.basename(path)
        rows.append({"filename": filename, "dire_score": round(prob, 4), "is_generated": prob > 0.5})
        print(f"{filename}: {prob:.4f}")

with open("dire_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "dire_score", "is_generated"])
    writer.writeheader()
    writer.writerows(rows)
print("\nWrote dire_results.csv")

## 8. Download the results

Drop the downloaded `dire_results.csv` anywhere in your local project and point `DIRE_RESULTS_CSV` at it:
```bash
export DIRE_RESULTS_CSV=/path/to/dire_results.csv
```
`audit/dire.py` looks up each image by filename in this CSV before trying (and failing, on a GPU-less machine) to run DIRE locally.

In [ ]:
from google.colab import files
files.download("dire_results.csv")